<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-03-arvores/Aula%2024%20-%20%C3%81rvore%20de%20Atingibilidade%20e%20Cobertura%20em%20Redes%20de%20Petri%20-%20code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Simulação e Análise Formal de Redes de Petri na Planta de Biodiesel (Grupo 5)
### Construção de Árvores de Atingibilidade e Algoritmo de Cobertura de Karp & Miller para o Reator CSTR R-200

**Unidade Operacional:** Setor 100 (Insumos TK-101/TK-103), Setor 200 (Reator CSTR R-200), Setor 300 (Decantador TK-300) e Setor 400 (Purificação)  
**Domínio de Engenharia:** Sistemas a Eventos Discretos (SED), Controle de Batelada (ISA-88), Redes de Petri Lugar/Transição (P/T Nets), Segurança Funcional (IEC 61511 / SIL 3)  

**Objetivos do Estudo:**
1. Modelar matematicamente o ciclo de batelada da síntese de biodiesel como uma Rede de Petri formal $\mathcal{N} = (P, T, F, W, M_0)$;
2. Implementar o algoritmo da **Árvore de Atingibilidade** e o algoritmo de **Árvore de Cobertura de Karp & Miller** com manipulação estrita do símbolo $\omega$;
3. Provar formalmente as propriedades comportamentais: **1-Limitabilidade (Segurança/Safeness)**, **Vivacidade ($L_4$)**, **Ausência de Deadlock** e **Reversibilidade**;
4. Calcular os invariantes estruturais fundamentais (**P-Invariantes** e **T-Invariantes**) e validar a conservação de recursos e exclusão mútua na decantação.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Set, Optional, Any, Union

# Configuração visual dos gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

print("✅ Módulos científicos, matriciais e de grafos importados com sucesso!")

---
## 1. Implementação da Rede de Petri e Manipulação do Símbolo $\omega$

O símbolo $\omega$ (*ômega*) denota uma quantidade arbitrariamente grande de fichas em redes de Petri, permitindo que o algoritmo de **Karp & Miller** represente espaços infinitos de marcações de forma estritamente finita.

Definimos uma representação aritmética rigorosa para $\omega$:
- $\omega > n, \quad \forall n \in \mathbb{N}$
- $\omega \pm n = \omega$
- $\omega \ge \omega$

In [ ]:
class OmegaValue:
    """Representa o símbolo matemático Ômega (ω) de Karp & Miller para marcas não-limitadas."""
    def __repr__(self):
        return "ω"
    
    def __str__(self):
        return "ω"
    
    def __eq__(self, other):
        return isinstance(other, OmegaValue)
    
    def __lt__(self, other):
        return False  # ω nunca é menor que nada
    
    def __gt__(self, other):
        return not isinstance(other, OmegaValue)  # ω > qualquer inteiro n
    
    def __ge__(self, other):
        return True  # ω >= qualquer valor, inclusive ω
    
    def __le__(self, other):
        return isinstance(other, OmegaValue)
    
    def __add__(self, other):
        return self
    
    def __radd__(self, other):
        return self
    
    def __sub__(self, other):
        return self
    
    def __hash__(self):
        return hash("OMEGA_INFINITY")

OMEGA = OmegaValue()

Token = Union[int, OmegaValue]
Marking = Tuple[Token, ...]

print(f"Símbolo Ômega definido: OMEGA = {OMEGA}. Teste OMEGA > 99999: {OMEGA > 99999}")

---
## 2. Classe `PetriNet`: Topologia, Matrizes e Regras de Disparo

Formalizamos a quíntupla $\mathcal{N} = (P, T, F, W, M_0)$:
- $C^-(p, t) = W(p, t)$ (Matriz de Pré-Incidência — consumo de fichas);
- $C^+(p, t) = W(t, p)$ (Matriz de Pós-Incidência — produção de fichas);
- $C = C^+ - C^-$ (Matriz de Incidência Combinada);
- Equação de Estado: $M' = M + C \cdot \mathbf{u}_t$.

In [ ]:
class PetriNet:
    """Modelagem formal de Redes de Petri Lugar/Transição (P/T Nets)."""
    def __init__(self, places: List[str], transitions: List[str]):
        self.places = list(places)
        self.transitions = list(transitions)
        self.p_idx = {p: i for i, p in enumerate(self.places)}
        self.t_idx = {t: i for i, t in enumerate(self.transitions)}
        
        self.num_p = len(self.places)
        self.num_t = len(self.transitions)
        
        # Matrizes de pré e pós incidência C- e C+
        self.pre_matrix = np.zeros((self.num_p, self.num_t), dtype=int)
        self.post_matrix = np.zeros((self.num_p, self.num_t), dtype=int)
        self.initial_marking: Marking = (0,) * self.num_p
        
    def add_input_arc(self, place: str, transition: str, weight: int = 1):
        """Arco direcionado Lugar -> Transição (Consumo de fichas)."""
        p = self.p_idx[place]
        t = self.t_idx[transition]
        self.pre_matrix[p, t] += weight
        
    def add_output_arc(self, transition: str, place: str, weight: int = 1):
        """Arco direcionado Transição -> Lugar (Produção de fichas)."""
        t = self.t_idx[transition]
        p = self.p_idx[place]
        self.post_matrix[p, t] += weight
        
    def set_initial_marking(self, marking: List[int]):
        assert len(marking) == self.num_p, "A marcação deve ter tamanho igual ao número de lugares!"
        self.initial_marking = tuple(marking)
        
    @property
    def incidence_matrix(self) -> np.ndarray:
        """Matriz de incidência combinada C = C+ - C-."""
        return self.post_matrix - self.pre_matrix
    
    def is_enabled(self, transition: str, marking: Marking) -> bool:
        """Verifica se a transição está habilitada: M(p) >= W(p, t) para todo p de entrada."""
        t = self.t_idx[transition]
        for p in range(self.num_p):
            w = self.pre_matrix[p, t]
            if w > 0:
                val = marking[p]
                if isinstance(val, OmegaValue):
                    continue
                if val < w:
                    return False
        return True
    
    def fire(self, transition: str, marking: Marking) -> Marking:
        """Calcula a nova marcação gerada pelo disparo da transição: M' = M - C-(., t) + C+(., t)."""
        if not self.is_enabled(transition, marking):
            raise ValueError(f"Transição {transition} não está habilitada na marcação {marking}!")
        t = self.t_idx[transition]
        new_m = list(marking)
        for p in range(self.num_p):
            if isinstance(new_m[p], OmegaValue):
                continue
            new_m[p] = new_m[p] - self.pre_matrix[p, t] + self.post_matrix[p, t]
        return tuple(new_m)
    
    def get_enabled_transitions(self, marking: Marking) -> List[str]:
        """Retorna a lista de transições atualmente habilitadas."""
        return [t for t in self.transitions if self.is_enabled(t, marking)]

print("✅ Classe PetriNet compilada com suporte matricial e regras de disparo.")

---
## 3. Algoritmo da Árvore de Cobertura de Karp & Miller

O algoritmo constrói a árvore enraizada em $M_0$ avaliando as relações de dominância ao longo de cada ramo ancestral:
1. Se $M'$ domina um nó ancestral $M''$ ($M' \ge M''$ e $M' \neq M''$), todo lugar onde $M'(p_k) > M''(p_k)$ recebe o símbolo $\omega$;
2. Se $M'$ é idêntico a um nó previamente existente na árvore, o nó é rotulado como **Duplicado** (fechamento de ciclo);
3. Se $M'$ não habilita nenhuma transição, é rotulado como **Terminal** (*Deadlock*).

In [ ]:
@dataclass
class TreeNode:
    node_id: int
    marking: Marking
    parent: Optional['TreeNode'] = None
    transition_from_parent: Optional[str] = None
    depth: int = 0
    is_terminal: bool = False
    is_duplicate: bool = False
    duplicate_of_id: Optional[int] = None
    children: List['TreeNode'] = field(default_factory=list)
    
    def is_ancestor(self, other: 'TreeNode') -> bool:
        curr = self.parent
        while curr is not None:
            if curr.node_id == other.node_id:
                return True
            curr = curr.parent
        return False

class CoverabilityTree:
    """Gerador determinístico da Árvore de Cobertura (Algoritmo de Karp & Miller)."""
    def __init__(self, net: PetriNet):
        self.net = net
        self.nodes: List[TreeNode] = []
        self.root: Optional[TreeNode] = None
        
    def _dominates(self, m1: Marking, m2: Marking) -> bool:
        """Retorna True se m1 domina estritamente m2: m1 >= m2 e m1 != m2."""
        greater_or_equal = True
        strictly_greater = False
        for v1, v2 in zip(m1, m2):
            if isinstance(v1, OmegaValue) and not isinstance(v2, OmegaValue):
                strictly_greater = True
            elif not isinstance(v1, OmegaValue) and isinstance(v2, OmegaValue):
                return False
            elif isinstance(v1, OmegaValue) and isinstance(v2, OmegaValue):
                continue
            else:
                if v1 < v2:
                    return False
                if v1 > v2:
                    strictly_greater = True
        return greater_or_equal and strictly_greater
    
    def build(self, max_depth: int = 50) -> 'CoverabilityTree':
        self.nodes = []
        self.root = TreeNode(node_id=0, marking=self.net.initial_marking, depth=0)
        self.nodes.append(self.root)
        
        unexplored: List[TreeNode] = [self.root]
        counter = 1
        
        while unexplored:
            curr = unexplored.pop(0)
            
            if curr.depth >= max_depth:
                continue
                
            enabled_t = self.net.get_enabled_transitions(curr.marking)
            if not enabled_t:
                curr.is_terminal = True
                continue
                
            for t in enabled_t:
                m_next = self.net.fire(t, curr.marking)
                
                # Regra de Karp & Miller: Verificar dominância em relação aos ancestrais
                ancestor = curr
                m_modified = list(m_next)
                while ancestor is not None:
                    if self._dominates(m_next, ancestor.marking):
                        for p in range(self.net.num_p):
                            v_next = m_next[p]
                            v_anc = ancestor.marking[p]
                            if (isinstance(v_next, OmegaValue) or isinstance(v_anc, OmegaValue)):
                                m_modified[p] = OMEGA
                            elif v_next > v_anc:
                                m_modified[p] = OMEGA
                    ancestor = ancestor.parent
                m_final = tuple(m_modified)
                
                # Verificar se a marcação já foi explorada na árvore
                existing_node = None
                for n in self.nodes:
                    if n.marking == m_final:
                        existing_node = n
                        break
                
                child = TreeNode(
                    node_id=counter,
                    marking=m_final,
                    parent=curr,
                    transition_from_parent=t,
                    depth=curr.depth + 1
                )
                counter += 1
                curr.children.append(child)
                self.nodes.append(child)
                
                if existing_node is not None:
                    child.is_duplicate = True
                    child.duplicate_of_id = existing_node.node_id
                else:
                    unexplored.append(child)
                    
        return self

print("✅ Gerador da Árvore de Cobertura de Karp & Miller implementado com sucesso!")

---
## 4. Modelagem da Planta Piloto de Biodiesel (Grupo 5)

Mapeamos os equipamentos e condições operacionais dos 4 setores da planta:

### Lugares ($P$):
1. `P_REATOR_VAZIO`: Reator R-200 pronto para nova batelada;
2. `P_OLEO_DISP`: Silo TK-101 com óleo vegetal degomado disponível;
3. `P_METOXIDO_DISP`: Tanque TK-103 com metóxido pronto;
4. `P_CARREGANDO_OLEO`: Válvula XV-201 aberta e bomba P-101 em transferência;
5. `P_DOSANDO_METOXIDO`: Válvula XV-202 aberta e bomba P-102 em dosagem;
6. `P_CARGA_COMPLETA`: Reagentes no reator (nível LT-201 em ~65%);
7. `P_AGITADOR_ATIVO`: Agitador AG-201 em operação contínua a 120 RPM;
8. `P_REACAO_AQUECIMENTO`: Jaqueta HT-201 energizada (temperatura 55 a 60 °C);
9. `P_BATELADA_CONCLUIDA`: Reação química concluída com conversão >= 96.5%;
10. `P_DECANTADOR_LIVRE`: Vaso decantador TK-300 descarregado e disponível;
11. `P_DECANTACAO_FASES`: Separação gravitacional bifásica em repouso;
12. `P_PURGA_GLICERINA`: Dreno inferior XV-301 aberto purgando glicerina pesada;
13. `P_TRANSF_BIODIESEL`: Transbordo XV-302 aberto enviando biodiesel à purificação.

### Transições ($T$):
- `t1`: `T_INICIA_CARGA_OLEO`
- `t2`: `T_FIM_CARGA_OLEO`
- `t3`: `T_INICIA_DOSAGEM_MET`
- `t4`: `T_FIM_DOSAGEM_MET`
- `t5`: `T_LIGA_AGITADOR_AQUECEDOR`
- `t6`: `T_FIM_REACAO`
- `t7`: `T_DESCARGA_DECANTADOR`
- `t8`: `T_FIM_SEPARACAO`
- `t9`: `T_INICIA_PURGA_GLIC`
- `t10`: `T_FIM_PURGA_GLIC`
- `t11`: `T_INICIA_TRANSF_BIO`
- `t12`: `T_CICLO_COMPLETO`

In [ ]:
places_biodiesel = [
    "P_REATOR_VAZIO",       # p0
    "P_OLEO_DISP",          # p1
    "P_METOXIDO_DISP",      # p2
    "P_CARREGANDO_OLEO",    # p3
    "P_DOSANDO_METOXIDO",   # p4
    "P_CARGA_COMPLETA",     # p5
    "P_AGITADOR_ATIVO",     # p6
    "P_REACAO_AQUECIMENTO", # p7
    "P_BATELADA_CONCLUIDA", # p8
    "P_DECANTADOR_LIVRE",   # p9
    "P_DECANTACAO_FASES",   # p10
    "P_PURGA_GLICERINA",    # p11
    "P_TRANSF_BIODIESEL"    # p12
]

transitions_biodiesel = [
    "T_INICIA_CARGA_OLEO",       # t0
    "T_FIM_CARGA_OLEO",          # t1
    "T_INICIA_DOSAGEM_MET",      # t2
    "T_FIM_DOSAGEM_MET",         # t3
    "T_LIGA_AGITADOR_AQUECEDOR", # t4
    "T_FIM_REACAO",              # t5
    "T_DESCARGA_DECANTADOR",     # t6
    "T_FIM_SEPARACAO",           # t7
    "T_INICIA_PURGA_GLIC",       # t8
    "T_FIM_PURGA_GLIC",          # t9
    "T_INICIA_TRANSF_BIO",       # t10
    "T_CICLO_COMPLETO"           # t11
]

net_biodiesel = PetriNet(places_biodiesel, transitions_biodiesel)

# Arcos de entrada e saída (Topologia do processo químico)
# t0: Inicia carga de óleo (consome reator vazio e óleo disponível)
net_biodiesel.add_input_arc("P_REATOR_VAZIO", "T_INICIA_CARGA_OLEO")
net_biodiesel.add_input_arc("P_OLEO_DISP", "T_INICIA_CARGA_OLEO")
net_biodiesel.add_output_arc("T_INICIA_CARGA_OLEO", "P_CARREGANDO_OLEO")

# t1: Fim da carga de óleo
net_biodiesel.add_input_arc("P_CARREGANDO_OLEO", "T_FIM_CARGA_OLEO")

# t2: Inicia dosagem de metóxido (requer metóxido disponível)
net_biodiesel.add_input_arc("P_METOXIDO_DISP", "T_INICIA_DOSAGEM_MET")
net_biodiesel.add_output_arc("T_FIM_CARGA_OLEO", "P_DOSANDO_METOXIDO")

# t3: Fim da dosagem de metóxido -> carga completa
net_biodiesel.add_input_arc("P_DOSANDO_METOXIDO", "T_FIM_DOSAGEM_MET")
net_biodiesel.add_output_arc("T_FIM_DOSAGEM_MET", "P_CARGA_COMPLETA")

# t4: Permissivo SIS: liga agitador e jaqueta térmica
net_biodiesel.add_input_arc("P_CARGA_COMPLETA", "T_LIGA_AGITADOR_AQUECEDOR")
net_biodiesel.add_output_arc("T_LIGA_AGITADOR_AQUECEDOR", "P_AGITADOR_ATIVO")
net_biodiesel.add_output_arc("T_LIGA_AGITADOR_AQUECEDOR", "P_REACAO_AQUECIMENTO")

# t5: Fim da reação CSTR (cinética concluída)
net_biodiesel.add_input_arc("P_AGITADOR_ATIVO", "T_FIM_REACAO")
net_biodiesel.add_input_arc("P_REACAO_AQUECIMENTO", "T_FIM_REACAO")
net_biodiesel.add_output_arc("T_FIM_REACAO", "P_BATELADA_CONCLUIDA")

# t6: Descarga para decantador TK-300 (requer decantador livre)
net_biodiesel.add_input_arc("P_BATELADA_CONCLUIDA", "T_DESCARGA_DECANTADOR")
net_biodiesel.add_input_arc("P_DECANTADOR_LIVRE", "T_DESCARGA_DECANTADOR")
net_biodiesel.add_output_arc("T_DESCARGA_DECANTADOR", "P_REATOR_VAZIO")  # Libera reator R-200
net_biodiesel.add_output_arc("T_DESCARGA_DECANTADOR", "P_DECANTACAO_FASES")

# t7: Fim da separação bifásica gravitacional no TK-300
net_biodiesel.add_input_arc("P_DECANTACAO_FASES", "T_FIM_SEPARACAO")

# t8: Inicia purga de glicerina pura (XV-301)
net_biodiesel.add_output_arc("T_FIM_SEPARACAO", "P_PURGA_GLICERINA")

# t9: Fim da purga de glicerina
net_biodiesel.add_input_arc("P_PURGA_GLICERINA", "T_FIM_PURGA_GLIC")

# t10: Inicia transferência de biodiesel bruto (XV-302) para purificação
net_biodiesel.add_output_arc("T_FIM_PURGA_GLIC", "P_TRANSF_BIODIESEL")

# t11: Ciclo completo: esvazia TK-300 e restabelece os insumos para nova batelada
net_biodiesel.add_input_arc("P_TRANSF_BIODIESEL", "T_CICLO_COMPLETO")
net_biodiesel.add_output_arc("T_CICLO_COMPLETO", "P_DECANTADOR_LIVRE")
net_biodiesel.add_output_arc("T_CICLO_COMPLETO", "P_OLEO_DISP")
net_biodiesel.add_output_arc("T_CICLO_COMPLETO", "P_METOXIDO_DISP")

# Marcação Inicial M0: Reator Vazio, Insumos Prontos e Decantador Livre
m0 = [1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
net_biodiesel.set_initial_marking(m0)

df_c = pd.DataFrame(net_biodiesel.incidence_matrix, index=places_biodiesel, columns=transitions_biodiesel)
print("Matriz de Incidência Combinada C (13 Lugares x 12 Transições):")
df_c

---
## 5. Construção e Exploração da Árvore de Cobertura

Executamos o algoritmo de Karp & Miller para derivar o grafo de cobertura completo da planta de transesterificação.

In [ ]:
tree_builder = CoverabilityTree(net_biodiesel)
tree_builder.build(max_depth=30)

print(f"Árvore de Cobertura construída com sucesso!")
print(f"Total de nós gerados na árvore: {len(tree_builder.nodes)}")

# Relatório tabular de todas as marcações alcançadas
tree_records = []
for n in tree_builder.nodes:
    status = "Raiz M0" if n.parent is None else ("Duplicado (Ciclo)" if n.is_duplicate else ("Deadlock" if n.is_terminal else "Explorado"))
    tree_records.append({
        "Nó ID": f"N_{n.node_id}",
        "Profundidade": n.depth,
        "Transição Origem": n.transition_from_parent or "---",
        "Marcação M": str(list(n.marking)),
        "Classificação": status,
        "Retorna a Nó": f"N_{n.duplicate_of_id}" if n.is_duplicate else "---"
    })

df_tree = pd.DataFrame(tree_records)
df_tree

---
## 6. Verificação Automática das Propriedades Formais

Análise rigorosa das propriedades provadas a partir da árvore construída:
1. **Limitabilidade e Segurança (*Safeness*):** Todos os lugares contêm no máximo 1 ficha (1-limitada);
2. **Ausência de Deadlock:** Não existem nós terminais sem transição habilitada fora do ciclo de fechamento;
3. **Reversibilidade:** A marcação inicial $M_0$ é restabelecida ao término do ciclo produtivo;
4. **Exclusão Mútua da Drenagem:** $M(\text{P_PURGA_GLICERINA}) + M(\text{P_TRANSF_BIODIESEL}) \le 1$ em todos os estados.

In [ ]:
all_markings = [n.marking for n in tree_builder.nodes]

# 1. Teste de Limitabilidade (Presença de Ômega)
has_omega = any(any(isinstance(v, OmegaValue) for v in m) for m in all_markings)
max_tokens = max(max(v for v in m if not isinstance(v, OmegaValue)) for m in all_markings)
is_safe = (not has_omega) and (max_tokens == 1)

# 2. Teste de Deadlock
deadlock_nodes = [n for n in tree_builder.nodes if n.is_terminal]
is_deadlock_free = len(deadlock_nodes) == 0

# 3. Teste de Reversibilidade (Ciclo fecha de volta em M0)
cycle_closed = any(n.is_duplicate and n.duplicate_of_id == 0 for n in tree_builder.nodes)

# 4. Teste de Exclusão Mútua (Purga de Glicerina vs Transferência de Biodiesel)
idx_purga = net_biodiesel.p_idx["P_PURGA_GLICERINA"]
idx_transf = net_biodiesel.p_idx["P_TRANSF_BIODIESEL"]
mutual_exclusion_violated = False

for m in all_markings:
    if m[idx_purga] == 1 and m[idx_transf] == 1:
        mutual_exclusion_violated = True
        break

print("=" * 65)
print("           RELATÓRIO DE AUDITORIA FORMAL (GRUPO 5)           ")
print("=" * 65)
print(f"1. Rede Limitada (Presença de ω)?      : {'NÃO (Limitada)' if not has_omega else 'SIM (Não-limitada)'}")
print(f"2. k-Limitabilidade Máxima             : k = {max_tokens} ficha(s)")
print(f"3. Rede Estritamente Segura (Safe / 1-limitada)?: {'SIM (Aprovada para CLP)' if is_safe else 'NÃO'}")
print(f"4. Ausência de Deadlock Global         : {'LIVRE DE DEADLOCK' if is_deadlock_free else 'FALHA'}")
print(f"5. Reversibilidade Operacional (M0)    : {'SIM (Ciclo Fechado)' if cycle_closed else 'NÃO'}")
print(f"6. Exclusão Mútua no Decantador TK-300 : {'GARANTIDA (Sem Contaminação)' if not mutual_exclusion_violated else 'VIOLADA'}")
print("=" * 65)

assert is_safe, "A rede deve ser estritamente 1-limitada para implementação direta em memórias de CLP!"
assert is_deadlock_free, "O processo de batelada não pode conter travamentos mútuos!"
assert cycle_closed, "A marcação inicial M0 deve ser periodicamente restaurada!"
assert not mutual_exclusion_violated, "Válvulas XV-301 e XV-302 nunca podem abrir concomitantemente!"

---
## 7. Análise Algébrica de Invariantes Estruturais (P-Invariants e T-Invariants)

Calculamos os vetores que preservam a conservação estequiométrica e a integridade de recursos:
- **P-Invariantes ($C^T \mathbf{y} = \mathbf{0}$):** Conservação física de fichas em sub-redes monousuárias;
- **T-Invariantes ($C \mathbf{x} = \mathbf{0}$):** Sequências periódicas de transições que realizam a batelada química.

In [ ]:
import scipy.linalg

C = net_biodiesel.incidence_matrix

# P-Invariantes: Vetores y >= 0 tais que y^T C = 0
# Oposto: Espaço nulo à esquerda de C
null_p = scipy.linalg.null_space(C.T)

# Normalizar base inteira simplificada dos P-invariantes
print(f"Dimensão do Espaço Nulo dos P-Invariantes: {null_p.shape[1]}")

# Invariante 1: Reator R-200 (p0 + p3 + p4 + p5 + p7 + p8 = 1)
y_reator = np.array([1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0])
res_reator = y_reator @ C
assert np.all(res_reator == 0), "y_reator deve ser um P-invariante exato!"

# Invariante 2: Vaso Decantador TK-300 (p9 + p10 + p11 + p12 = 1)
y_decant = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1])
res_decant = y_decant @ C
assert np.all(res_decant == 0), "y_decant deve ser um P-invariante exato!"

# T-Invariante Fundamental: x = [1, 1, ..., 1]^T
x_batch = np.ones(net_biodiesel.num_t, dtype=int)
res_t = C @ x_batch
assert np.all(res_t == 0), "O vetor unitário deve ser o T-invariante do ciclo de batelada!"

print("✅ Provas algébricas de P-Invariantes e T-Invariantes validadas com sucesso:")
print("  • Invariante Físico do Reator CSTR R-200 : y_reator^T * C = 0")
print("  • Invariante Físico do Decantador TK-300  : y_decant^T * C = 0")
print("  • T-Invariante Periódico de Batelada     : C * x_batch = 0")

---
## 8. Visualização Gráfica: Rede de Petri e Árvore de Cobertura Hierárquica

Renderizamos:
1. O **Grafo Bipartido da Rede de Petri** com diferenciação estética entre Lugares e Transições;
2. A **Árvore de Cobertura Gerada**, evidenciando o caminho da raiz $M_0$ até o fechamento cíclico.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# ----------------- PAINEL 1: GRAFO BIPARTIDO DA REDE DE PETRI -----------------
ax1 = axes[0]
G_petri = nx.DiGraph()

for p in places_biodiesel:
    G_petri.add_node(p, bipartite=0, tipo="lugar")
for t in transitions_biodiesel:
    G_petri.add_node(t, bipartite=1, tipo="transicao")

for p in places_biodiesel:
    for t in transitions_biodiesel:
        p_i = net_biodiesel.p_idx[p]
        t_j = net_biodiesel.t_idx[t]
        if net_biodiesel.pre_matrix[p_i, t_j] > 0:
            G_petri.add_edge(p, t)
        if net_biodiesel.post_matrix[p_i, t_j] > 0:
            G_petri.add_edge(t, p)

pos_petri = nx.spring_layout(G_petri, seed=42, k=0.85)

lugar_nodes = [n for n, d in G_petri.nodes(data=True) if d["tipo"] == "lugar"]
trans_nodes = [n for n, d in G_petri.nodes(data=True) if d["tipo"] == "transicao"]

nx.draw_networkx_nodes(G_petri, pos_petri, nodelist=lugar_nodes, node_color='#38bdf8', node_shape='o', node_size=900, edgecolors='#0284c7', linewidths=2.0, ax=ax1)
nx.draw_networkx_nodes(G_petri, pos_petri, nodelist=trans_nodes, node_color='#fb7185', node_shape='s', node_size=800, edgecolors='#e11d48', linewidths=2.0, ax=ax1)
nx.draw_networkx_edges(G_petri, pos_petri, edge_color='#64748b', arrowsize=14, width=1.5, ax=ax1)

# Rótulos simplificados para legibilidade
labels_petri = {p: f"P{i}" for i, p in enumerate(places_biodiesel)}
labels_petri.update({t: f"t{i+1}" for i, t in enumerate(transitions_biodiesel)})
nx.draw_networkx_labels(G_petri, pos_petri, labels=labels_petri, font_size=9, font_weight='bold', font_color='#0f172a', ax=ax1)

ax1.set_title("Grafo Bipartido da Rede de Petri (Batelada R-200 / TK-300)\n(Círculos Azuis = Lugares P; Quadrados Vermelhos = Transições t)", fontsize=11, fontweight='bold', pad=12)
ax1.axis('off')

# ----------------- PAINEL 2: ÁRVORE DE COBERTURA (KARP & MILLER) -----------------
ax2 = axes[1]
G_tree = nx.DiGraph()

color_map = []
labels_tree = {}

for n in tree_builder.nodes:
    node_name = f"M{n.node_id}"
    G_tree.add_node(node_name)
    labels_tree[node_name] = f"M_{n.node_id}\n(d={n.depth})"
    
    if n.node_id == 0:
        color_map.append('#10b981')  # Verde esmeralda para raiz M0
    elif n.is_duplicate:
        color_map.append('#f59e0b')  # Âmbar para ciclo de retorno
    elif n.is_terminal:
        color_map.append('#ef4444')  # Vermelho para deadlock (se houver)
    else:
        color_map.append('#60a5fa')  # Azul claro para estados regulares
        
    if n.parent is not None:
        p_name = f"M{n.parent.node_id}"
        t_short = f"t_{net_biodiesel.t_idx[n.transition_from_parent] + 1}"
        G_tree.add_edge(p_name, node_name, label=t_short)

# Layout em camadas por profundidade
pos_tree = {}
depth_groups = {}
for n in tree_builder.nodes:
    depth_groups.setdefault(n.depth, []).append(n)

for d, group in depth_groups.items():
    for idx, node in enumerate(group):
        pos_tree[f"M{node.node_id}"] = (idx - len(group)/2.0, -d * 1.5)

nx.draw_networkx_nodes(G_tree, pos_tree, node_color=color_map, node_size=1200, edgecolors='#334155', linewidths=2.0, ax=ax2)
nx.draw_networkx_edges(G_tree, pos_tree, edge_color='#475569', arrowsize=16, width=2.0, ax=ax2)
nx.draw_networkx_labels(G_tree, pos_tree, labels=labels_tree, font_size=8, font_weight='bold', font_color='#ffffff', ax=ax2)

edge_labels = nx.get_edge_attributes(G_tree, 'label')
nx.draw_networkx_edge_labels(G_tree, pos_tree, edge_labels=edge_labels, font_size=8, font_color='#dc2626', ax=ax2)

ax2.set_title("Árvore de Cobertura de Karp & Miller\n(Verde = M₀; Azul = Estados Válidos; Âmbar = Fechamento de Ciclo)", fontsize=11, fontweight='bold', pad=12)
ax2.axis('off')

plt.tight_layout()
plt.show()

---
## 9. Simulação Cronológica e Diagrama de Estados da Batelada

Simulamos o disparo sequencial de todas as transições da batelada e mapeamos a cronologia das etapas operacionais no Reator R-200 e no Decantador TK-300.

In [ ]:
curr_m = net_biodiesel.initial_marking
history = [curr_m]
fired_log = []

for t in transitions_biodiesel:
    assert net_biodiesel.is_enabled(t, curr_m), f"Transição {t} deveria estar habilitada!"
    curr_m = net_biodiesel.fire(t, curr_m)
    history.append(curr_m)
    fired_log.append(t)

mat_hist = np.array(history)

plt.figure(figsize=(14, 6))
plt.imshow(mat_hist.T, aspect='auto', cmap='Blues', interpolation='nearest')
plt.colorbar(label='Fichas no Lugar (0 ou 1)', ticks=[0, 1])
plt.yticks(range(len(places_biodiesel)), places_biodiesel, fontsize=9)
plt.xticks(range(len(history)), [f"Passo {i}" for i in range(len(history))], fontsize=9)
plt.title("Mapa de Ativação Temporal dos Lugares ao Longo da Batelada Química (Grupo 5)", fontsize=12, fontweight='bold', pad=12)
plt.xlabel("Evolução dos Passos de Disparo (t₁ a t₁₂)", fontsize=10, fontweight='bold')
plt.ylabel("Lugares da Rede de Petri (ISA-88)", fontsize=10, fontweight='bold')

# Anotar os nomes das transições sobre o gráfico
for i, t in enumerate(fired_log):
    plt.text(i + 0.5, -0.7, f"t{i+1}", ha='center', va='bottom', fontsize=8, color='#b91c1c', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Batelada completa simulada com reversibilidade perfeita: Marcação final idêntica a M0!")

---
## 10. Conclusões Técnicas e Diretrizes de Engenharia

1. **Verificação Formal e Imunidade a Deadlocks:** A árvore de cobertura comprovou que a lógica de batelada da planta de biodiesel não possui estados bloqueantes, operando em ciclo fechado reversível;
2. **Implementação Determinística em CLP (IEC 61131-3):** A prova de 1-limitabilidade assegura que cada lugar pode ser traduzido diretamente para etapas de SFC (*Sequential Function Chart*) com mapeamento de memória booleano de 1 bit, eliminando qualquer risco de estouro de variáveis;
3. **Garantia de Exclusão Mútua:** Os invariantes estruturais e a árvore de estados comprovaram que a purga de glicerina e a transferência de biodiesel nunca ocorrem simultaneamente, resguardando a qualidade do produto final e a segurança operacional dos setores 200, 300 e 400.